# Turning weight-space programming on its head: quickstart

This notebook gets the repository, plays some of the published networks, checks the original experiment's
networks against their records, rebuilds the results tables from the saved reads, and trains a network.

In Colab, choose **Runtime > Change runtime type > GPU** first, then **Runtime > Run all**. It takes about five
minutes. Everything also runs without a GPU, more slowly.

It uses the versions of JAX and NumPy that Colab already has, and installs only what Colab lacks. Scores
therefore match the ones quoted here to within rounding, not to the last digit. For the exact environment the
networks were trained in, use the `Dockerfile` in the repository.

In [ ]:
!git clone --depth 1 https://github.com/letapjar/weight-space-verification.git
%cd weight-space-verification/code

In [ ]:
!pip -q install flax optax distrax

In [ ]:
import jax
print("JAX", jax.__version__, "on", jax.devices()[0])

## Play published networks

Each file in `networks/` is one of the networks the models were asked to read, named by a random id. The answer
key says `93b4f9dc` is a Breakout network. It should score about 22.8.

In [ ]:
!python -m jaxport.play_network ../networks/93b4f9dc.npz breakout

The same network with its board turned 180°, and another Breakout network with all 500 of its inputs permuted.
Each plays its own version of the game as well as an unchanged network plays Breakout: about 22.8 and 22.6.

In [ ]:
!python -m jaxport.play_network ../networks/711c3b8f.npz breakout_turned180
!python -m jaxport.play_network ../networks/b585d2d8.npz breakout_shuffled1

A mixed network: Breakout's first layer on Pong's later layers. The models named it Breakout. It scores about
0.6, the level of random play.

In [ ]:
!python -m jaxport.play_network ../networks/5df3d4e0.npz breakout

## The original experiment's networks at our games

Our games were rebuilt from the original's description and recordings. Each line shows the original's network
playing our version of its game, beside the score in its own training record.

In [ ]:
!python -m jaxport.check_originals

## Rebuild the results tables from the saved reads

This reads every reply in `reads/` and rewrites `scoring/results.md` and `scoring/results.csv`. The second
command prints nothing if the rebuilt tables are identical to the published ones.

In [ ]:
!python results_table.py
!git status --short ../scoring

## Train a network

The recipe of the twelve networks in the main table, cut from 200 million steps to 20 million. On a GPU this
takes under a minute and reaches a score of about 22. Without a GPU, change `--steps` to `2000000`.

In [ ]:
!python -m jaxport.ppo --game breakout --activation tanh --seeds 0 --copies 256 --learning-rate 0.005 --steps 20000000 --runs-dir runs_quickstart

## The tests (optional, a few minutes)

In [ ]:
!python -m pytest -q jaxport test_readings.py